In [ ]:
## Load required libraries
!pip install -q transformers datasets sentencepiece sacremoses evaluate accelerate


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback
)
import evaluate


In [ ]:
## Check GPU availability
## Set device configuration

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


In [ ]:
## Load dataset
dataset = load_dataset("cfilt/iitb-english-hindi")


In [ ]:
## Load dataset
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(50000))
dataset["validation"] = dataset["validation"].shuffle(seed=42).select(range(520))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(2500))


In [ ]:
## login to hugging face
!pip install -q huggingface_hub


In [ ]:
from huggingface_hub import login
login()


In [ ]:
model_name = "ai4bharat/indictrans2-en-indic-dist-200M"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True
).to(device)


In [ ]:
# Disable cache (prevents crash)
model.config.use_cache = False

# Save GPU memory
model.gradient_checkpointing_enable()


In [ ]:
## Preprocess dataset for translation
SRC_LANG = "eng_Latn"
TGT_LANG = "hin_Deva"

max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = [
        f"{SRC_LANG} {TGT_LANG} {item['en']}"
        for item in examples["translation"]
    ]
    targets = [item["hi"] for item in examples["translation"]]

    model_inputs = tokenizer(
        inputs,
        truncation=True,
        padding="max_length",
        max_length=max_input_length
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            truncation=True,
            padding="max_length",
            max_length=max_target_length
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [ ]:
## Load pre-trained IndicTrans2 model and tokenizer
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)


In [ ]:
## Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/indictrans2_finetune",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,
    weight_decay=0.01,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,   # effective batch = 64

    num_train_epochs=4,
    fp16=True,

    predict_with_generate=False,
    prediction_loss_only=True,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,
    report_to="none"
)


In [ ]:
# Define early stopping
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=1,
    early_stopping_threshold=0.01
)


In [ ]:
## Start model fine-tuning
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[early_stopping]
)


In [ ]:
##empty cache necessary for indictrans2
torch.cuda.empty_cache()
trainer.train()


In [ ]:
#evaluate loss
test_results = trainer.evaluate(tokenized_datasets["test"])

print(test_results)


In [ ]:
#checking which check point to push
load_best_model_at_end=True
metric_for_best_model="eval_loss"


In [ ]:
##push mode to hugging face for  testing
final_model_dir = "/kaggle/working/final_model"

trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)


In [ ]:
training_args.hub_model_id = REPO_ID


In [ ]:
trainer.push_to_hub(
    commit_message="Fine-tuned IndicTrans2 on IITB En-Hi dataset (Kaggle, 20k samples, 4 epochs)"
)
